# OPF PII Benchmarks — Colab Runner

This notebook reproduces the [opf-benchmarks](https://github.com/CupOfGeo/opf-benchmarks-geo) suite: OpenAI Privacy Filter (OPF) evaluated against the three benchmarks NVIDIA published GLiNER-PII numbers on (Argilla PII, AI4Privacy, Nemotron-PII), in three modes each.

**Before running:** switch to a GPU runtime via *Runtime → Change runtime type → T4 GPU*. CPU works but takes much longer.

Then *Runtime → Run all*. The headline F1 table renders in the last cell.

In [ ]:
import os

REPO_DIR = "opf-benchmarks-geo"
if not os.path.isdir(REPO_DIR):
    !git clone https://github.com/CupOfGeo/opf-benchmarks-geo.git
%cd {REPO_DIR}

In [ ]:
!pip install -q .

In [ ]:
import torch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")
if DEVICE == "cpu":
    print("WARNING: no GPU detected — runs will be slow. Switch to T4 via Runtime → Change runtime type.")

## Knobs

Tweak these to change the run. `MAX_EXAMPLES=1000` finishes in ~10–15 min on a free Colab T4 — bump to `5000` to match the sample size in the project README.

In [ ]:
MAX_EXAMPLES = 1000
BENCHMARKS = ["argilla", "ai4privacy", "nemotron"]

In [ ]:
!python -m scripts.download_datasets --max-examples {MAX_EXAMPLES} --benchmarks {" ".join(BENCHMARKS)}

In [ ]:
!python -m opf_benchmarks.run_eval --device {DEVICE} --benchmarks {" ".join(BENCHMARKS)}

In [ ]:
!python -m opf_benchmarks.aggregate

## Results

In [ ]:
from IPython.display import Markdown

Markdown(open("results/REPORT.md").read())

## (Optional) Download the report

Run the cell below in Colab to download `REPORT.md` to your machine. Skipped automatically when not running in Colab.

In [ ]:
try:
    from google.colab import files
    files.download("results/REPORT.md")
except ImportError:
    print("Not running in Colab — skipping download.")